# Notebook 06 — Policy-First Optuna and Breadth Feature-Set Selection

## Purpose

Use the frozen Stage 05 purged anchored walk-forward folds to select a
train-only Random Forest or XGBoost decision-support model after integrating
the five Stage 04 additions:

- `started_run_length`;
- `market_breadth_raw`;
- `market_breadth_ema30`;
- `market_breadth_slope5`;
- `market_breadth_regime`.

### Non-negotiable controls

- Candidate population remains the frozen 118,464 Stage 04 long events.
- Stage 05 folds are consumed exactly and are never redesigned.
- Average Uniqueness is computed fold-locally within symbol.
- The unseen test is not opened.
- No probability threshold is selected.
- Daily signal diagnostics use the pre-registered Top 5% rule with at least
  one signal per date.
- `started` and Breadth are model features only; no hard filter is applied.

### Two-stage selection

1. Tune Random Forest and XGBoost on the complete 40-feature superset.
   Optuna still receives equal-fold mean ROC AUC as its sampler objective.
2. Select each model family's trial using the fixed daily policy hierarchy:
   minimize false positives first, then maximize precision and fold stability.
3. With the selected family hyperparameters frozen, compare nine
   pre-registered feature sets under the same daily policy.
4. Select the final model and feature set with false-positive reduction as
   the primary objective.

Using shared family hyperparameters avoids running a separate Optuna search
for every feature subset and limits search multiplicity.


In [1]:
from __future__ import annotations

from pathlib import Path
import gc
import json
import sys
import time

import numpy as np
import optuna
import pandas as pd
import yaml

print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("optuna:", optuna.__version__)


e:\Iran_stock_trade\financial-ml-decision-support\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.12.2
numpy: 1.26.4
pandas: 2.2.3
optuna: 4.2.1


## 1. Locate repository and import project modules

In [2]:
def locate_repository_root(start: Path) -> Path:
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "configs").exists()
            and (candidate / "notebooks").exists()
            and (candidate / "src").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Repository root was not found. "
        "Open the project root in VS Code."
    )


REPOSITORY_ROOT = locate_repository_root(Path.cwd())

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.evaluation.classification_metrics import (
    binary_classification_metrics,
)
from src.models.baselines import (
    build_dummy_prior_pipeline,
    build_logistic_regression_pipeline,
)
from src.models.policy_selection import (
    POLICY_SELECTION_SCHEMA_VERSION,
    DailyTopFractionPolicy,
    apply_daily_top_fraction_policy,
    rank_policy_candidates,
    select_optuna_trial_by_policy,
    summarize_policy_predictions,
)
from src.models.random_forest import (
    build_random_forest_pipeline,
    sample_random_forest_params,
)
from src.models.tuning import (
    fold_auc_summary,
)
from src.models.xgboost_model import (
    build_xgboost_pipeline,
    sample_xgboost_params,
)
from src.utils.paths import (
    data_paths,
    repository_result_paths,
    resolve_data_root,
)
from src.utils.reproducibility import (
    git_commit_sha,
    software_manifest,
    stable_object_hash,
)
from src.validation.folds import WalkForwardFold
from src.validation.purged_walk_forward import (
    fold_membership_masks,
    normalize_candidate_event_panel,
)
from src.validation.sample_weighting import (
    AVERAGE_UNIQUENESS_SCHEMA_VERSION,
    compute_fold_train_average_uniqueness,
    summarize_fold_average_uniqueness,
)

print("Repository root:", REPOSITORY_ROOT)


Repository root: E:\Iran_stock_trade\financial-ml-decision-support


## 2. Load frozen Stage 04/05 artifacts and Stage 06 v4 policy configuration


In [3]:
def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = yaml.safe_load(file_obj)

    if not isinstance(value, dict):
        raise ValueError(
            f"Configuration must be a mapping: {path}"
        )

    return value


paths_config = load_yaml(
    REPOSITORY_ROOT / "configs" / "paths.yaml"
)
optuna_config = load_yaml(
    REPOSITORY_ROOT / "configs" / "optuna.yaml"
)
models_config = load_yaml(
    REPOSITORY_ROOT / "configs" / "models.yaml"
)
stage06_config = load_yaml(
    REPOSITORY_ROOT
    / "configs"
    / "stage06_breadth_retrain.yaml"
)

DATA_ROOT = resolve_data_root(
    paths_config,
    REPOSITORY_ROOT,
)
DATA_PATHS = data_paths(
    DATA_ROOT,
    paths_config,
)
RESULT_PATHS = repository_result_paths(
    REPOSITORY_ROOT,
    paths_config,
)

CANDIDATE_TRAIN_PATH = (
    DATA_PATHS["candidate_data"] / "train"
)
LABELED_TRAIN_PATH = (
    DATA_PATHS["labeled_data"] / "train"
)

feature_manifest_path = (
    RESULT_PATHS["manifests"]
    / "04_approved_model_features.csv"
)
stage04_policy_path = (
    RESULT_PATHS["manifests"]
    / "04_candidate_filter_policy.json"
)
stage05_manifest_path = (
    RESULT_PATHS["manifests"]
    / "05_purged_anchored_walk_forward_manifest.json"
)
fold_boundaries_path = (
    RESULT_PATHS["folds"]
    / "05_purged_anchored_walk_forward_boundaries.csv"
)
validation_assignments_path = (
    RESULT_PATHS["folds"]
    / "05_validation_event_assignments.csv"
)

for required_path in [
    feature_manifest_path,
    stage04_policy_path,
    stage05_manifest_path,
    fold_boundaries_path,
    validation_assignments_path,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            "Required frozen-stage artifact is missing: "
            f"{required_path}"
        )

feature_manifest_df = pd.read_csv(
    feature_manifest_path,
    low_memory=False,
).sort_values(
    "feature_order",
    kind="stable",
).reset_index(drop=True)

stage04_policy = json.loads(
    stage04_policy_path.read_text(encoding="utf-8")
)
stage05_manifest = json.loads(
    stage05_manifest_path.read_text(encoding="utf-8")
)
fold_boundaries_df = pd.read_csv(
    fold_boundaries_path,
    low_memory=False,
)
validation_assignments_df = pd.read_csv(
    validation_assignments_path,
    low_memory=False,
)

MODEL_FEATURES = (
    feature_manifest_df["feature"].astype(str).tolist()
)
NUMERIC_FEATURES = (
    feature_manifest_df.loc[
        feature_manifest_df["data_type"].eq("numeric"),
        "feature",
    ].astype(str).tolist()
)
CATEGORICAL_FEATURES = (
    feature_manifest_df.loc[
        feature_manifest_df["data_type"].eq("categorical"),
        "feature",
    ].astype(str).tolist()
)

ADDED_STAGE04_FEATURES = list(
    stage06_config["added_stage04_features"].keys()
)
BASELINE_FEATURES = [
    feature
    for feature in MODEL_FEATURES
    if feature not in ADDED_STAGE04_FEATURES
]

FEATURE_SETS = {}
FEATURE_SET_NUMERIC = {}
FEATURE_SET_CATEGORICAL = {}

for feature_set_name, additions in (
    stage06_config["feature_sets"].items()
):
    requested = set(BASELINE_FEATURES) | set(additions)
    feature_list = [
        feature
        for feature in MODEL_FEATURES
        if feature in requested
    ]
    if len(feature_list) != len(requested):
        missing = sorted(requested - set(feature_list))
        raise KeyError(
            f"{feature_set_name} contains unknown features: {missing}"
        )
    FEATURE_SETS[feature_set_name] = feature_list
    FEATURE_SET_NUMERIC[feature_set_name] = [
        feature
        for feature in feature_list
        if feature in NUMERIC_FEATURES
    ]
    FEATURE_SET_CATEGORICAL[feature_set_name] = [
        feature
        for feature in feature_list
        if feature in CATEGORICAL_FEATURES
    ]

TUNING_FEATURE_SET_NAME = str(
    stage06_config["design"]["tuning_feature_set"]
)
TUNING_FEATURES = FEATURE_SETS[
    TUNING_FEATURE_SET_NAME
]
TUNING_NUMERIC_FEATURES = FEATURE_SET_NUMERIC[
    TUNING_FEATURE_SET_NAME
]
TUNING_CATEGORICAL_FEATURES = FEATURE_SET_CATEGORICAL[
    TUNING_FEATURE_SET_NAME
]

SEED = int(
    optuna_config["reproducibility"]["random_seed"]
)
N_COMPLETE_TRIALS = int(
    optuna_config["optimization"][
        "required_complete_trials_per_model"
    ]
)
DIAGNOSTIC_THRESHOLD = float(
    optuna_config["objective"][
        "probability_threshold_for_diagnostics"
    ]
)

policy_config = stage06_config["policy"]
PRIMARY_POLICY = DailyTopFractionPolicy(
    fraction=float(
        policy_config["selected_fraction"]
    ),
    minimum_signals_per_date=int(
        policy_config["minimum_signals_per_date"]
    ),
)

WEIGHTED_MODELS = set(
    models_config["sample_weighting"]["applied_models"]
)

assert stage06_config["status"] == (
    "stage_06_breadth_retrain_v4"
)
assert optuna_config["status"] == (
    "stage_06_configured_v3"
)
assert models_config["status"] == (
    "stage_06_configured_v3"
)
assert stage05_manifest["status"] == (
    "frozen_after_integrity_checks"
)
assert stage05_manifest["unseen_test_used"] is False
assert stage05_manifest[
    "fold_design_schema_version"
] == (
    "stage05_v2_candidate_event_count_balanced_contiguous"
)
assert stage04_policy["primary_side"] == "long"
assert np.isclose(
    float(
        stage04_policy[
            "primary_threshold_fraction"
        ]
    ),
    0.15,
)

assert len(MODEL_FEATURES) == 40
assert len(NUMERIC_FEATURES) == 38
assert CATEGORICAL_FEATURES == [
    "gmma_state",
    "market_breadth_regime",
]
assert len(BASELINE_FEATURES) == 35
assert set(ADDED_STAGE04_FEATURES) == {
    "started_run_length",
    "market_breadth_raw",
    "market_breadth_ema30",
    "market_breadth_slope5",
    "market_breadth_regime",
}
assert len(FEATURE_SETS) == 9
assert TUNING_FEATURE_SET_NAME == "I_full_40"
assert TUNING_FEATURES == MODEL_FEATURES
assert N_COMPLETE_TRIALS == 30
assert np.isclose(PRIMARY_POLICY.fraction, 0.05)
assert PRIMARY_POLICY.minimum_signals_per_date == 1

assert AVERAGE_UNIQUENESS_SCHEMA_VERSION == (
    models_config["sample_weighting"][
        "schema_version"
    ]
)
assert models_config["sample_weighting"][
    "validation_events_used_to_compute_weights"
] is False
assert models_config["sample_weighting"][
    "validation_metrics_weighted"
] is False
assert models_config["sample_weighting"][
    "dummy_prior_weighted"
] is False
assert WEIGHTED_MODELS == {
    "logistic_regression",
    "random_forest",
    "xgboost",
}

assert optuna_config["pruner"]["name"] == (
    "NopPruner"
)
assert optuna_config["pruner"][
    "pruning_enabled"
] is False
assert optuna_config["optimization"][
    "every_trial_evaluates_all_folds"
] is True
assert optuna_config["objective"]["formula"] == (
    "mean_fold_roc_auc"
)

print("Model features:", len(MODEL_FEATURES))
print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", CATEGORICAL_FEATURES)
print("Baseline features:", len(BASELINE_FEATURES))
print("Pre-registered feature sets:", len(FEATURE_SETS))
print("Tuning feature set:", TUNING_FEATURE_SET_NAME)
print(
    "Required COMPLETE trials per tuned model:",
    N_COMPLETE_TRIALS,
)
print(
    "Fixed daily policy:",
    f"Top {PRIMARY_POLICY.fraction:.0%}, "
    f"minimum {PRIMARY_POLICY.minimum_signals_per_date}",
)
print("Threshold selected: False")
print("Unseen-test used: False")

display(
    pd.DataFrame(
        [
            {
                "feature_set_name": name,
                "features": len(features),
                "numeric_features": len(
                    FEATURE_SET_NUMERIC[name]
                ),
                "categorical_features": len(
                    FEATURE_SET_CATEGORICAL[name]
                ),
                "added_features": [
                    feature
                    for feature in features
                    if feature in ADDED_STAGE04_FEATURES
                ],
            }
            for name, features in FEATURE_SETS.items()
        ]
    )
)


Model features: 40
Numeric features: 38
Categorical features: ['gmma_state', 'market_breadth_regime']
Baseline features: 35
Pre-registered feature sets: 9
Tuning feature set: I_full_40
Required COMPLETE trials per tuned model: 30
Fixed daily policy: Top 5%, minimum 1
Threshold selected: False
Unseen-test used: False


,feature_set_name,features,numeric_features,categorical_features,added_features
0,A_baseline_35,35,34,1,[]
1,B_plus_started,36,35,1,[started_run_length]
2,C_plus_breadth_raw,36,35,1,[market_breadth_raw]
3,D_plus_breadth_ema30,36,35,1,[market_breadth_ema30]
4,E_plus_breadth_slope5,36,35,1,[market_breadth_slope5]
5,F_plus_all_continuous_breadth,38,37,1,"[market_breadth_raw, market_breadth_ema30, mar..."
6,G_plus_breadth_regime,36,34,2,[market_breadth_regime]
7,H_plus_slope5_and_regime,37,35,2,"[market_breadth_slope5, market_breadth_regime]"
8,I_full_40,40,38,2,"[started_run_length, market_breadth_raw, marke..."


## 3. Reconstruct the exact frozen Stage 05 v2 folds

In [4]:
def boundary_row_to_fold(row) -> WalkForwardFold:
    return WalkForwardFold(
        fold_id=int(row.fold_id),
        calendar_start_date=pd.Timestamp(
            row.calendar_start_date
        ),
        train_end_date=pd.Timestamp(
            row.train_end_date
        ),
        embargo_start_date=pd.Timestamp(
            row.embargo_start_date
        ),
        embargo_end_date=pd.Timestamp(
            row.embargo_end_date
        ),
        validation_start_date=pd.Timestamp(
            row.validation_start_date
        ),
        validation_end_date=pd.Timestamp(
            row.validation_end_date
        ),
        train_end_calendar_index=int(
            row.train_end_calendar_index
        ),
        validation_start_calendar_index=int(
            row.validation_start_calendar_index
        ),
        validation_end_calendar_index=int(
            row.validation_end_calendar_index
        ),
        embargo_trading_days=int(
            row.embargo_trading_days
        ),
        validation_target_candidate_events=float(
            row.validation_target_candidate_events
        ),
        validation_candidate_events=int(
            row.validation_candidate_events
        ),
        validation_event_count_deviation=float(
            row.validation_event_count_deviation
        ),
        validation_event_count_relative_deviation=float(
            row.validation_event_count_relative_deviation
        ),
        validation_horizon_candidate_events=int(
            row.validation_horizon_candidate_events
        ),
        validation_partition_method=str(
            row.validation_partition_method
        ),
    )


folds = [
    boundary_row_to_fold(row)
    for row in fold_boundaries_df.itertuples(
        index=False
    )
]

assert len(folds) == 5
assert [
    fold.fold_id
    for fold in folds
] == [1, 2, 3, 4, 5]
assert all(
    fold.validation_partition_method
    == (
        "candidate_event_count_balanced_"
        "contiguous_calendar"
    )
    for fold in folds
)

display(fold_boundaries_df)


,fold_id,calendar_start_date,train_end_date,embargo_start_date,embargo_end_date,validation_start_date,validation_end_date,train_end_calendar_index,validation_start_calendar_index,validation_end_calendar_index,embargo_trading_days,validation_target_candidate_events,validation_candidate_events,validation_event_count_deviation,validation_event_count_relative_deviation,validation_horizon_candidate_events,validation_partition_method
0,1,2014-03-19,2017-08-06,2017-08-07,2017-09-18,2017-09-19,2018-01-31,814,845,936,30,10368.0,10344,-24.0,-0.002315,51840,candidate_event_count_balanced_contiguous_cale...
1,2,2014-03-19,2017-12-20,2017-12-23,2018-01-31,2018-02-03,2018-06-27,906,937,1027,30,10368.0,10408,40.0,0.003858,51840,candidate_event_count_balanced_contiguous_cale...
2,3,2014-03-19,2018-05-12,2018-05-13,2018-06-27,2018-06-30,2019-01-15,997,1028,1165,30,10368.0,10397,29.0,0.002797,51840,candidate_event_count_balanced_contiguous_cale...
3,4,2014-03-19,2018-12-04,2018-12-05,2019-01-15,2019-01-16,2020-01-04,1135,1166,1396,30,10368.0,10319,-49.0,-0.004726,51840,candidate_event_count_balanced_contiguous_cale...
4,5,2014-03-19,2019-11-23,2019-11-24,2020-01-04,2020-01-05,2021-03-17,1366,1397,1689,30,10368.0,10372,4.0,0.000386,51840,candidate_event_count_balanced_contiguous_cale...


## 4. Build the pooled Stage 04 40-feature candidate-event panel


In [5]:
candidate_files = sorted(
    CANDIDATE_TRAIN_PATH.glob(
        "*_train_candidates.csv"
    )
)

if len(candidate_files) != int(
    stage05_manifest[
        "candidate_population"
    ]["symbols"]
):
    raise AssertionError(
        "Candidate-file count no longer matches Stage 05."
    )


def candidate_symbol_from_path(path: Path) -> str:
    suffix = "_train_candidates.csv"

    if not path.name.endswith(suffix):
        raise ValueError(
            f"Unexpected candidate file: {path.name}"
        )

    return path.name[:-len(suffix)]


candidate_parts = []
candidate_error_rows = []
started = time.perf_counter()

usecols = [
    "dEven",
    "event_end_date",
    "meta_label",
    *MODEL_FEATURES,
]

for file_number, path in enumerate(
    candidate_files,
    start=1,
):
    symbol = candidate_symbol_from_path(path)

    try:
        frame = pd.read_csv(
            path,
            usecols=usecols,
            low_memory=False,
        )
        frame.insert(0, "symbol", symbol)

        event_dates = pd.to_datetime(
            frame["dEven"],
            errors="coerce",
        )
        frame.insert(
            0,
            "event_id",
            (
                symbol
                + "::"
                + event_dates.dt.strftime("%Y-%m-%d")
            ),
        )
        candidate_parts.append(frame)

    except Exception as exc:
        candidate_error_rows.append(
            {
                "symbol": symbol,
                "file_name": path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 50 == 0
        or file_number == len(candidate_files)
    ):
        elapsed = time.perf_counter() - started
        print(
            f"[candidate panel] "
            f"[{file_number:>4}/{len(candidate_files)}] "
            f"elapsed={elapsed:,.1f}s "
            f"errors={len(candidate_error_rows)}"
        )

candidate_errors_df = pd.DataFrame(
    candidate_error_rows,
    columns=[
        "symbol",
        "file_name",
        "error_type",
        "error_message",
    ],
)

candidate_errors_df.to_csv(
    RESULT_PATHS["audits"]
    / "06_candidate_panel_errors.csv",
    index=False,
    encoding="utf-8-sig",
)

if not candidate_errors_df.empty:
    raise RuntimeError(
        "Candidate panel contains loading errors. "
        "Review 06_candidate_panel_errors.csv"
    )

candidate_panel = normalize_candidate_event_panel(
    pd.concat(
        candidate_parts,
        ignore_index=True,
    )
)

for feature in NUMERIC_FEATURES:
    candidate_panel[feature] = pd.to_numeric(
        candidate_panel[feature],
        errors="coerce",
    )

for feature in CATEGORICAL_FEATURES:
    candidate_panel[feature] = (
        candidate_panel[feature].astype("string")
    )

candidate_panel = candidate_panel.sort_values(
    ["symbol", "dEven", "event_id"],
    kind="stable",
).reset_index(drop=True)

expected_events = int(
    stage05_manifest[
        "candidate_population"
    ]["events"]
)

assert len(candidate_panel) == expected_events
assert candidate_panel["event_id"].is_unique
assert candidate_panel["meta_label"].isin(
    [0, 1]
).all()
assert set(MODEL_FEATURES).issubset(
    candidate_panel.columns
)

numeric_array = candidate_panel[
    NUMERIC_FEATURES
].to_numpy(dtype=float)

assert not np.isinf(numeric_array).any()

panel_summary_df = pd.DataFrame(
    [
        {
            "candidate_events": len(candidate_panel),
            "symbols": int(
                candidate_panel[
                    "symbol"
                ].nunique()
            ),
            "features": len(MODEL_FEATURES),
            "numeric_features": len(
                NUMERIC_FEATURES
            ),
            "categorical_features": len(
                CATEGORICAL_FEATURES
            ),
            "feature_sets": len(FEATURE_SETS),
            "positive_fraction": float(
                candidate_panel[
                    "meta_label"
                ].mean()
            ),
            "numeric_missing_values": int(
                candidate_panel[
                    NUMERIC_FEATURES
                ].isna().sum().sum()
            ),
            "categorical_missing_values": int(
                candidate_panel[
                    CATEGORICAL_FEATURES
                ].isna().sum().sum()
            ),
            "duplicate_event_ids": int(
                candidate_panel[
                    "event_id"
                ].duplicated().sum()
            ),
            "hard_started_filter_applied": False,
            "hard_breadth_filter_applied": False,
            "unseen_test_used": False,
        }
    ]
)

panel_summary_df.to_csv(
    RESULT_PATHS["audits"]
    / "06_candidate_panel_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

display(panel_summary_df)


[candidate panel] [   1/499] elapsed=0.1s errors=0
[candidate panel] [  50/499] elapsed=2.6s errors=0
[candidate panel] [ 100/499] elapsed=4.8s errors=0
[candidate panel] [ 150/499] elapsed=7.1s errors=0
[candidate panel] [ 200/499] elapsed=9.4s errors=0
[candidate panel] [ 250/499] elapsed=11.2s errors=0
[candidate panel] [ 300/499] elapsed=13.2s errors=0
[candidate panel] [ 350/499] elapsed=15.4s errors=0
[candidate panel] [ 400/499] elapsed=17.3s errors=0
[candidate panel] [ 450/499] elapsed=19.2s errors=0
[candidate panel] [ 499/499] elapsed=21.1s errors=0


,candidate_events,symbols,features,numeric_features,categorical_features,feature_sets,positive_fraction,numeric_missing_values,categorical_missing_values,duplicate_event_ids,hard_started_filter_applied,hard_breadth_filter_applied,unseen_test_used
0,118464,499,40,38,2,9,0.539176,22379,1353,0,False,False,False


## 5. Reproduce frozen Stage 05 validation membership exactly

In [6]:
validation_assignments_df["event_id"] = (
    validation_assignments_df[
        "event_id"
    ].astype(str)
)
validation_assignments_df["fold_id"] = pd.to_numeric(
    validation_assignments_df["fold_id"],
    errors="raise",
).astype(int)

fold_reproduction_rows = []
fold_position_cache = {}

for fold in folds:
    masks = fold_membership_masks(
        candidate_panel,
        fold,
    )

    train_positions = np.flatnonzero(
        masks.train.to_numpy(dtype=bool)
    )
    validation_positions = np.flatnonzero(
        masks.validation.to_numpy(dtype=bool)
    )

    fold_position_cache[fold.fold_id] = {
        "train": train_positions,
        "validation": validation_positions,
    }

    reproduced_ids = set(
        candidate_panel.iloc[
            validation_positions
        ]["event_id"].astype(str)
    )
    frozen_ids = set(
        validation_assignments_df.loc[
            validation_assignments_df[
                "fold_id"
            ].eq(fold.fold_id),
            "event_id",
        ].astype(str)
    )

    missing_from_reproduction = (
        frozen_ids - reproduced_ids
    )
    unexpected_in_reproduction = (
        reproduced_ids - frozen_ids
    )

    fold_reproduction_rows.append(
        {
            "fold_id": fold.fold_id,
            "frozen_validation_events": len(
                frozen_ids
            ),
            "reproduced_validation_events": len(
                reproduced_ids
            ),
            "reproduced_train_events": int(
                len(train_positions)
            ),
            "missing_from_reproduction": len(
                missing_from_reproduction
            ),
            "unexpected_in_reproduction": len(
                unexpected_in_reproduction
            ),
            "exact_membership_match": (
                not missing_from_reproduction
                and not unexpected_in_reproduction
            ),
        }
    )

fold_reproduction_audit_df = pd.DataFrame(
    fold_reproduction_rows
)

fold_reproduction_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "06_fold_reproduction_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

assert fold_reproduction_audit_df[
    "exact_membership_match"
].all()

display(fold_reproduction_audit_df)


,fold_id,frozen_validation_events,reproduced_validation_events,reproduced_train_events,missing_from_reproduction,unexpected_in_reproduction,exact_membership_match
0,1,10344,10344,62717,0,0,True
1,2,10408,10408,73590,0,0,True
2,3,10397,10397,83825,0,0,True
3,4,10319,10319,94059,0,0,True
4,5,10372,10372,107360,0,0,True


## 6. Load symbol-specific labeled-train calendars

Average Uniqueness is evaluated over each security's own sequence of trading
observations. Different securities never contribute to one another's
concurrency.


In [7]:
def labeled_symbol_from_path(path: Path) -> str:
    suffix = "_train_labeled.csv"

    if not path.name.endswith(suffix):
        raise ValueError(
            f"Unexpected labeled-train file: {path.name}"
        )

    return path.name[:-len(suffix)]


labeled_train_files = sorted(
    LABELED_TRAIN_PATH.glob(
        "*_train_labeled.csv"
    )
)

calendar_error_rows = []
symbol_calendars = {}
calendar_audit_rows = []

started = time.perf_counter()

for file_number, path in enumerate(
    labeled_train_files,
    start=1,
):
    symbol = labeled_symbol_from_path(path)

    try:
        date_frame = pd.read_csv(
            path,
            usecols=["dEven"],
            low_memory=False,
        )

        calendar = pd.DatetimeIndex(
            sorted(
                pd.to_datetime(
                    date_frame["dEven"],
                    errors="coerce",
                )
                .dropna()
                .dt.normalize()
                .unique()
            )
        )

        if calendar.empty:
            raise ValueError(
                "Labeled-train calendar is empty."
            )

        if calendar.has_duplicates:
            raise AssertionError(
                "Labeled-train calendar has duplicates."
            )

        symbol_calendars[symbol] = calendar

        calendar_audit_rows.append(
            {
                "symbol": symbol,
                "calendar_observations": int(
                    len(calendar)
                ),
                "first_date": calendar.min(),
                "last_date": calendar.max(),
                "invalid_date_rows": int(
                    pd.to_datetime(
                        date_frame["dEven"],
                        errors="coerce",
                    ).isna().sum()
                ),
            }
        )

    except Exception as exc:
        calendar_error_rows.append(
            {
                "symbol": symbol,
                "file_name": path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 50 == 0
        or file_number == len(
            labeled_train_files
        )
    ):
        elapsed = time.perf_counter() - started
        print(
            f"[symbol calendars] "
            f"[{file_number:>4}/"
            f"{len(labeled_train_files)}] "
            f"elapsed={elapsed:,.1f}s "
            f"errors={len(calendar_error_rows)}"
        )

calendar_errors_df = pd.DataFrame(
    calendar_error_rows,
    columns=[
        "symbol",
        "file_name",
        "error_type",
        "error_message",
    ],
)
calendar_audit_df = pd.DataFrame(
    calendar_audit_rows
)

calendar_errors_df.to_csv(
    RESULT_PATHS["audits"]
    / "06_symbol_calendar_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
calendar_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "06_symbol_calendar_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

if not calendar_errors_df.empty:
    raise RuntimeError(
        "Symbol calendar errors exist. "
        "Review 06_symbol_calendar_errors.csv"
    )

candidate_symbols = set(
    candidate_panel["symbol"].astype(str)
)
calendar_symbols = set(symbol_calendars)

assert candidate_symbols == calendar_symbols
assert len(symbol_calendars) == 499

display(calendar_audit_df.describe(include="all"))


[symbol calendars] [   1/499] elapsed=0.2s errors=0
[symbol calendars] [  50/499] elapsed=2.1s errors=0
[symbol calendars] [ 100/499] elapsed=4.0s errors=0
[symbol calendars] [ 150/499] elapsed=6.0s errors=0
[symbol calendars] [ 200/499] elapsed=8.2s errors=0
[symbol calendars] [ 250/499] elapsed=10.1s errors=0
[symbol calendars] [ 300/499] elapsed=12.0s errors=0
[symbol calendars] [ 350/499] elapsed=13.7s errors=0
[symbol calendars] [ 400/499] elapsed=15.9s errors=0
[symbol calendars] [ 450/499] elapsed=17.8s errors=0
[symbol calendars] [ 499/499] elapsed=19.7s errors=0


,symbol,calendar_observations,first_date,last_date,invalid_date_rows
count,499,499.000000,499,499,499.0
unique,499,NaN,NaN,NaN,NaN
top,آ س پ,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN
mean,NaN,1320.194389,2014-08-12 21:01:04.929859584,2021-03-16 14:28:37.034067968,0.0
min,NaN,600.000000,2014-03-19 00:00:00,2021-02-09 00:00:00,0.0
25%,NaN,1122.500000,2014-03-19 00:00:00,2021-03-17 00:00:00,0.0
50%,NaN,1437.000000,2014-03-19 00:00:00,2021-03-17 00:00:00,0.0
75%,NaN,1550.500000,2014-04-08 00:00:00,2021-03-17 00:00:00,0.0
max,NaN,1638.000000,2018-07-16 00:00:00,2021-03-17 00:00:00,0.0


## 7. Compute Fold-Local Average Uniqueness weights

For each fold, only retained training events are supplied to the concurrency
calculation. Validation events are excluded before weighting begins.

The event interval is inclusive of both `dEven` and `event_end_date`. Raw
uniqueness is normalized to mean one inside the current fold's training set.


In [8]:
fold_training_weight_cache = {}
weight_event_parts = []
weight_symbol_audit_parts = []
weight_summary_rows = []

for fold in folds:
    train_positions = fold_position_cache[
        fold.fold_id
    ]["train"]
    validation_positions = fold_position_cache[
        fold.fold_id
    ]["validation"]

    train_events = candidate_panel.iloc[
        train_positions
    ][
        [
            "event_id",
            "symbol",
            "dEven",
            "event_end_date",
        ]
    ].copy()

    validation_event_ids = set(
        candidate_panel.iloc[
            validation_positions
        ]["event_id"].astype(str)
    )

    fold_weights, symbol_weight_audit = (
        compute_fold_train_average_uniqueness(
            train_events,
            symbol_calendars,
        )
    )

    weight_event_ids = set(
        fold_weights["event_id"].astype(str)
    )

    validation_ids_used = len(
        weight_event_ids.intersection(
            validation_event_ids
        )
    )

    if validation_ids_used != 0:
        raise AssertionError(
            "Validation events entered fold-training "
            "weight construction."
        )

    if weight_event_ids != set(
        train_events["event_id"].astype(str)
    ):
        raise AssertionError(
            "Average Uniqueness output does not exactly "
            "cover the fold-training events."
        )

    weight_by_event_id = (
        fold_weights.set_index(
            "event_id"
        )["sample_weight"]
    )

    aligned_weight = (
        train_events["event_id"]
        .astype(str)
        .map(weight_by_event_id)
        .to_numpy(dtype=float)
    )

    if not np.isfinite(aligned_weight).all():
        raise AssertionError(
            "Aligned fold-training weights are non-finite."
        )

    if not np.isclose(
        float(aligned_weight.mean()),
        1.0,
        atol=1.0e-12,
        rtol=1.0e-12,
    ):
        raise AssertionError(
            "Aligned fold-training weights do not "
            "average to one."
        )

    fold_training_weight_cache[
        fold.fold_id
    ] = aligned_weight

    fold_weights.insert(
        0,
        "fold_id",
        int(fold.fold_id),
    )
    weight_event_parts.append(fold_weights)

    symbol_weight_audit.insert(
        0,
        "fold_id",
        int(fold.fold_id),
    )
    weight_symbol_audit_parts.append(
        symbol_weight_audit
    )

    summary = summarize_fold_average_uniqueness(
        fold_weights,
        fold_id=fold.fold_id,
        validation_events_used=(
            validation_ids_used
        ),
    )
    summary.update(
        {
            "train_end_date": fold.train_end_date,
            "validation_start_date": (
                fold.validation_start_date
            ),
            "validation_end_date": (
                fold.validation_end_date
            ),
            "validation_events": int(
                len(validation_positions)
            ),
        }
    )
    weight_summary_rows.append(summary)

    print(
        f"fold={fold.fold_id} "
        f"train_events={len(train_events):,} "
        f"mean_raw_uniqueness="
        f"{summary['mean_raw_average_uniqueness']:.6f} "
        f"effective_sample_fraction="
        f"{summary['effective_sample_fraction']:.6f} "
        f"validation_events_used=0"
    )

all_fold_train_weights_df = pd.concat(
    weight_event_parts,
    ignore_index=True,
)
average_uniqueness_symbol_audit_df = pd.concat(
    weight_symbol_audit_parts,
    ignore_index=True,
)
average_uniqueness_audit_df = pd.DataFrame(
    weight_summary_rows
)

all_fold_train_weights_df.to_csv(
    RESULT_PATHS["folds"]
    / "06_fold_train_average_uniqueness_weights.csv",
    index=False,
    encoding="utf-8-sig",
)

average_uniqueness_symbol_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "06_average_uniqueness_symbol_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

average_uniqueness_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "06_average_uniqueness_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

display(average_uniqueness_audit_df)


fold=1 train_events=62,717 mean_raw_uniqueness=0.096185 effective_sample_fraction=0.488331 validation_events_used=0
fold=2 train_events=73,590 mean_raw_uniqueness=0.094140 effective_sample_fraction=0.490944 validation_events_used=0
fold=3 train_events=83,825 mean_raw_uniqueness=0.093830 effective_sample_fraction=0.489600 validation_events_used=0
fold=4 train_events=94,059 mean_raw_uniqueness=0.097678 effective_sample_fraction=0.482659 validation_events_used=0
fold=5 train_events=107,360 mean_raw_uniqueness=0.101322 effective_sample_fraction=0.470654 validation_events_used=0


,fold_id,train_events,symbols,minimum_raw_average_uniqueness,mean_raw_average_uniqueness,median_raw_average_uniqueness,maximum_raw_average_uniqueness,minimum_sample_weight,mean_sample_weight,maximum_sample_weight,...,nonfinite_sample_weight,nonpositive_sample_weight,validation_events_used,weight_source_scope,concurrency_scope,normalization,train_end_date,validation_start_date,validation_end_date,validation_events
0,1,62717,443,0.032258,0.096185,0.066131,1.0,0.335374,1.0,10.396599,...,0,0,0,current_fold_training_events_only,within_symbol,fold_train_mean_one,2017-08-06,2017-09-19,2018-01-31,10344
1,2,73590,469,0.032258,0.094140,0.064847,1.0,0.342661,1.0,10.622498,...,0,0,0,current_fold_training_events_only,within_symbol,fold_train_mean_one,2017-12-20,2018-02-03,2018-06-27,10408
2,3,83825,486,0.032258,0.093830,0.064593,1.0,0.343793,1.0,10.657585,...,0,0,0,current_fold_training_events_only,within_symbol,fold_train_mean_one,2018-05-12,2018-06-30,2019-01-15,10397
3,4,94059,495,0.032258,0.097678,0.066142,1.0,0.330248,1.0,10.237683,...,0,0,0,current_fold_training_events_only,within_symbol,fold_train_mean_one,2018-12-04,2019-01-16,2020-01-04,10319
4,5,107360,499,0.032258,0.101322,0.067397,1.0,0.318373,1.0,9.869563,...,0,0,0,current_fold_training_events_only,within_symbol,fold_train_mean_one,2019-11-23,2020-01-05,2021-03-17,10372


## 8. Frozen fold evaluation with selectable feature sets

Preprocessing, imputation, categorical encoding, and Average Uniqueness
weighting remain fold-local. A feature-set name controls only which approved
Stage 04 columns are supplied to the model.


In [9]:
def xgb_scale_pos_weight(
    y_train: np.ndarray,
    sample_weight: np.ndarray,
    mode: str,
) -> float:
    if mode == "none":
        return 1.0

    if mode != "fold_weighted_ratio":
        raise ValueError(
            f"Unknown class_weight_mode: {mode}"
        )

    y_array = np.asarray(
        y_train,
        dtype=int,
    )
    weight_array = np.asarray(
        sample_weight,
        dtype=float,
    )

    positive_mass = float(
        weight_array[y_array == 1].sum()
    )
    negative_mass = float(
        weight_array[y_array == 0].sum()
    )

    if positive_mass <= 0.0 or negative_mass <= 0.0:
        raise ValueError(
            "Both weighted classes are required in "
            "fold training data."
        )

    return negative_mass / positive_mass


def fit_predict_fold(
    *,
    model_name: str,
    feature_set_name: str,
    fold: WalkForwardFold,
    params: dict | None = None,
    return_predictions: bool = True,
) -> tuple[dict, pd.DataFrame | None]:
    if feature_set_name not in FEATURE_SETS:
        raise KeyError(
            f"Unknown feature set: {feature_set_name}"
        )

    model_features = FEATURE_SETS[
        feature_set_name
    ]
    numeric_features = FEATURE_SET_NUMERIC[
        feature_set_name
    ]
    categorical_features = FEATURE_SET_CATEGORICAL[
        feature_set_name
    ]

    positions = fold_position_cache[
        fold.fold_id
    ]

    train = candidate_panel.iloc[
        positions["train"]
    ]
    validation = candidate_panel.iloc[
        positions["validation"]
    ]

    X_train = train[model_features]
    y_train = train[
        "meta_label"
    ].to_numpy(dtype=int)

    X_validation = validation[
        model_features
    ]
    y_validation = validation[
        "meta_label"
    ].to_numpy(dtype=int)

    train_weight = fold_training_weight_cache[
        fold.fold_id
    ]

    if len(train_weight) != len(train):
        raise AssertionError(
            "Fold-training sample-weight length mismatch."
        )

    fold_seed = SEED + int(fold.fold_id)

    if model_name == "dummy_prior":
        model = build_dummy_prior_pipeline(
            numeric_features,
            categorical_features,
            seed=fold_seed,
        )

    elif model_name == "logistic_regression":
        model = build_logistic_regression_pipeline(
            numeric_features,
            categorical_features,
            seed=fold_seed,
        )

    elif model_name == "random_forest":
        if params is None:
            raise ValueError(
                "Random Forest params are required."
            )

        model = build_random_forest_pipeline(
            numeric_features,
            categorical_features,
            params=params,
            seed=fold_seed,
        )

    elif model_name == "xgboost":
        if params is None:
            raise ValueError(
                "XGBoost params are required."
            )

        scale_pos_weight = xgb_scale_pos_weight(
            y_train,
            train_weight,
            str(
                params.get(
                    "class_weight_mode",
                    "none",
                )
            ),
        )

        model = build_xgboost_pipeline(
            numeric_features,
            categorical_features,
            params=params,
            seed=fold_seed,
            scale_pos_weight=scale_pos_weight,
        )

    else:
        raise ValueError(
            f"Unknown model_name: {model_name}"
        )

    fit_started = time.perf_counter()

    if model_name in WEIGHTED_MODELS:
        model.fit(
            X_train,
            y_train,
            model__sample_weight=train_weight,
        )
        sample_weighting_applied = True
    else:
        model.fit(
            X_train,
            y_train,
        )
        sample_weighting_applied = False

    fit_seconds = (
        time.perf_counter() - fit_started
    )

    inference_started = time.perf_counter()
    probability = model.predict_proba(
        X_validation
    )[:, 1]
    inference_seconds = (
        time.perf_counter() - inference_started
    )

    metrics = binary_classification_metrics(
        y_validation,
        probability,
        threshold=DIAGNOSTIC_THRESHOLD,
    )
    metrics.update(
        {
            "model_name": model_name,
            "feature_set_name": feature_set_name,
            "feature_count": int(
                len(model_features)
            ),
            "numeric_feature_count": int(
                len(numeric_features)
            ),
            "categorical_feature_count": int(
                len(categorical_features)
            ),
            "fold_id": int(fold.fold_id),
            "train_events": int(len(train)),
            "validation_events": int(
                len(validation)
            ),
            "train_positive_fraction": float(
                y_train.mean()
            ),
            "validation_positive_fraction": float(
                y_validation.mean()
            ),
            "average_uniqueness_weighting_applied": (
                sample_weighting_applied
            ),
            "train_sample_weight_mean": (
                float(train_weight.mean())
                if sample_weighting_applied
                else 1.0
            ),
            "train_sample_weight_min": (
                float(train_weight.min())
                if sample_weighting_applied
                else 1.0
            ),
            "train_sample_weight_max": (
                float(train_weight.max())
                if sample_weighting_applied
                else 1.0
            ),
            "validation_metrics_weighted": False,
            "fit_seconds": float(
                fit_seconds
            ),
            "inference_seconds": float(
                inference_seconds
            ),
        }
    )

    predictions = None

    if return_predictions:
        predictions = validation[
            [
                "event_id",
                "symbol",
                "dEven",
                "event_end_date",
                "meta_label",
            ]
        ].copy()
        predictions.insert(
            0,
            "fold_id",
            int(fold.fold_id),
        )
        predictions.insert(
            0,
            "feature_set_name",
            feature_set_name,
        )
        predictions.insert(
            0,
            "model_name",
            model_name,
        )
        predictions[
            "probability_positive"
        ] = probability
        predictions[
            "diagnostic_prediction_at_0_50"
        ] = (
            probability
            >= DIAGNOSTIC_THRESHOLD
        ).astype(int)
        predictions[
            "validation_metric_weight"
        ] = 1.0

    del model
    gc.collect()

    return metrics, predictions


## 9. Evaluate fixed baselines

In [10]:
baseline_fold_metric_rows = []
baseline_prediction_parts = []

for model_name in [
    "dummy_prior",
    "logistic_regression",
]:
    print(f"\nBaseline: {model_name}")

    for fold in folds:
        metrics, predictions = fit_predict_fold(
            model_name=model_name,
            feature_set_name=TUNING_FEATURE_SET_NAME,
            fold=fold,
            return_predictions=True,
        )

        baseline_fold_metric_rows.append(
            metrics
        )
        baseline_prediction_parts.append(
            predictions
        )

        print(
            f"  fold={fold.fold_id} "
            f"roc_auc={metrics['roc_auc']:.6f} "
            f"specificity@0.50="
            f"{metrics['specificity']:.6f} "
            f"sensitivity@0.50="
            f"{metrics['sensitivity']:.6f} "
            f"weighted="
            f"{metrics['average_uniqueness_weighting_applied']}"
        )

baseline_fold_metrics_df = pd.DataFrame(
    baseline_fold_metric_rows
)
baseline_predictions_df = pd.concat(
    baseline_prediction_parts,
    ignore_index=True,
)

display(baseline_fold_metrics_df)



Baseline: dummy_prior
  fold=1 roc_auc=0.500000 specificity@0.50=0.000000 sensitivity@0.50=1.000000 weighted=False
  fold=2 roc_auc=0.500000 specificity@0.50=1.000000 sensitivity@0.50=0.000000 weighted=False
  fold=3 roc_auc=0.500000 specificity@0.50=1.000000 sensitivity@0.50=0.000000 weighted=False
  fold=4 roc_auc=0.500000 specificity@0.50=1.000000 sensitivity@0.50=0.000000 weighted=False
  fold=5 roc_auc=0.500000 specificity@0.50=0.000000 sensitivity@0.50=1.000000 weighted=False

Baseline: logistic_regression
  fold=1 roc_auc=0.436184 specificity@0.50=0.349701 sensitivity@0.50=0.541103 weighted=True
  fold=2 roc_auc=0.460055 specificity@0.50=0.207530 sensitivity@0.50=0.738728 weighted=True
  fold=3 roc_auc=0.692315 specificity@0.50=0.735325 sensitivity@0.50=0.519597 weighted=True
  fold=4 roc_auc=0.416922 specificity@0.50=0.206061 sensitivity@0.50=0.708533 weighted=True
  fold=5 roc_auc=0.597725 specificity@0.50=0.046362 sensitivity@0.50=0.964829 weighted=True


,events,positive_fraction,roc_auc,average_precision,log_loss,brier_score,threshold,accuracy,balanced_accuracy,precision,...,validation_events,train_positive_fraction,validation_positive_fraction,average_uniqueness_weighting_applied,train_sample_weight_mean,train_sample_weight_min,train_sample_weight_max,validation_metrics_weighted,fit_seconds,inference_seconds
0,10344,0.385731,0.500000,0.385731,0.694196,0.250524,0.5,0.385731,0.500000,0.385731,...,10344,0.502272,0.385731,False,1.0,1.000000,1.000000,False,0.848047,0.040436
1,10408,0.466660,0.500000,0.466660,0.691644,0.249249,0.5,0.533340,0.500000,0.000000,...,10408,0.485637,0.466660,False,1.0,1.000000,1.000000,False,0.891363,0.044358
2,10397,0.726363,0.500000,0.726363,0.717401,0.262116,0.5,0.273637,0.500000,0.000000,...,10397,0.474656,0.726363,False,1.0,1.000000,1.000000,False,0.960940,0.035174
3,10319,0.888071,0.500000,0.888071,0.695918,0.251385,0.5,0.111929,0.500000,0.000000,...,10319,0.498219,0.888071,False,1.0,1.000000,1.000000,False,1.028212,0.030075
4,10372,0.507135,0.500000,0.507135,0.695120,0.250983,0.5,0.507135,0.500000,0.507135,...,10372,0.539288,0.507135,False,1.0,1.000000,1.000000,False,1.268442,0.033394
5,10344,0.385731,0.436184,0.352304,0.777574,0.289462,0.5,0.423531,0.445402,0.343189,...,10344,0.502272,0.385731,True,1.0,0.335374,10.396599,False,5.172826,0.052609
6,10408,0.466660,0.460055,0.441355,0.734480,0.269641,0.5,0.455419,0.473129,0.449230,...,10408,0.485637,0.466660,True,1.0,0.342661,10.622498,False,3.532986,0.035364
7,10397,0.726363,0.692315,0.844339,0.679959,0.242505,0.5,0.578628,0.627461,0.838999,...,10397,0.474656,0.726363,True,1.0,0.343793,10.657585,False,3.141077,0.044349
8,10319,0.888071,0.416922,0.861176,0.634682,0.221559,0.5,0.652292,0.457297,0.876248,...,10319,0.498219,0.888071,True,1.0,0.330248,10.237683,False,5.577266,0.041431
9,10372,0.507135,0.597725,0.601174,0.926509,0.324531,0.5,0.512148,0.505595,0.510050,...,10372,0.539288,0.507135,True,1.0,0.318373,9.869563,False,6.188829,0.034672


## 10. Create versioned Optuna studies

Both model families are tuned on the complete 40-feature superset. Optuna uses
equal-fold mean ROC AUC for sampler guidance, while the final COMPLETE trial
selection applies the fixed Top 5% false-positive-first hierarchy.


In [11]:
study_signature_payload = {
    "feature_manifest": (
        feature_manifest_df.to_dict(
            orient="records"
        )
    ),
    "feature_sets": FEATURE_SETS,
    "tuning_feature_set": (
        TUNING_FEATURE_SET_NAME
    ),
    "stage06_config": stage06_config,
    "fold_boundaries": (
        fold_boundaries_df.to_dict(
            orient="records"
        )
    ),
    "optuna_config": optuna_config,
    "models_config": models_config,
    "stage04_policy": stage04_policy,
    "stage05_configuration_hash": (
        stage05_manifest[
            "configuration_hash"
        ]
    ),
    "average_uniqueness_schema": (
        AVERAGE_UNIQUENESS_SCHEMA_VERSION
    ),
    "policy_selection_schema": (
        POLICY_SELECTION_SCHEMA_VERSION
    ),
    "average_uniqueness_audit": (
        average_uniqueness_audit_df.to_dict(
            orient="records"
        )
    ),
}

STUDY_SIGNATURE = stable_object_hash(
    study_signature_payload
)[:12]

database_path = (
    REPOSITORY_ROOT
    / optuna_config[
        "storage"
    ]["database_file"]
)
database_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

storage_url = (
    "sqlite:///"
    + database_path.resolve().as_posix()
)


def build_study_sampler():
    return optuna.samplers.TPESampler(
        seed=SEED,
        multivariate=bool(
            optuna_config[
                "sampler"
            ]["multivariate"]
        ),
    )


def build_study_pruner():
    return optuna.pruners.NopPruner()


study_names = {
    "random_forest": (
        "stage06_v4_policy_random_forest_"
        f"{STUDY_SIGNATURE}"
    ),
    "xgboost": (
        "stage06_v4_policy_xgboost_"
        f"{STUDY_SIGNATURE}"
    ),
}

print("Study signature:", STUDY_SIGNATURE)
print("Optuna database:", database_path)
print("Study names:", study_names)
print("Tuning feature set:", TUNING_FEATURE_SET_NAME)


Study signature: 3ac1ee3f73df
Optuna database: E:\Iran_stock_trade\financial-ml-decision-support\results\optuna\06_optuna_studies.sqlite3
Study names: {'random_forest': 'stage06_v4_policy_random_forest_3ac1ee3f73df', 'xgboost': 'stage06_v4_policy_xgboost_3ac1ee3f73df'}
Tuning feature set: I_full_40


## 11. Define the complete five-fold objective

Every trial evaluates all five frozen folds and generates pooled out-of-fold
predictions. Optuna receives mean fold ROC AUC as the search objective, but each
trial also stores fixed Top 5% policy metrics. The selected COMPLETE trial is the
one with the fewest false positives, followed by precision, fold stability,
Average Precision, ROC AUC, and lower trial number.


In [12]:
def create_objective(model_name: str):
    if model_name not in {
        "random_forest",
        "xgboost",
    }:
        raise ValueError(
            "Only Random Forest and XGBoost are tuned."
        )

    def objective(
        trial: optuna.Trial,
    ) -> float:
        params = (
            sample_random_forest_params(
                trial
            )
            if model_name == "random_forest"
            else sample_xgboost_params(
                trial
            )
        )

        fold_metric_rows = []
        prediction_parts = []

        for fold in folds:
            metrics, predictions = fit_predict_fold(
                model_name=model_name,
                feature_set_name=(
                    TUNING_FEATURE_SET_NAME
                ),
                fold=fold,
                params=params,
                return_predictions=True,
            )
            fold_metric_rows.append(metrics)
            prediction_parts.append(predictions)

        if len(fold_metric_rows) != len(folds):
            raise AssertionError(
                "A trial did not evaluate all frozen folds."
            )

        auc_summary = fold_auc_summary(
            [
                row["roc_auc"]
                for row in fold_metric_rows
            ]
        )
        mean_average_precision = float(
            np.mean(
                [
                    row["average_precision"]
                    for row in fold_metric_rows
                ]
            )
        )

        trial_predictions = pd.concat(
            prediction_parts,
            ignore_index=True,
        )
        policy_predictions = (
            apply_daily_top_fraction_policy(
                trial_predictions,
                policy=PRIMARY_POLICY,
            )
        )
        policy_summary, policy_fold_metrics = (
            summarize_policy_predictions(
                policy_predictions,
                policy=PRIMARY_POLICY,
            )
        )

        trial.set_user_attr(
            "fold_metrics",
            fold_metric_rows,
        )
        trial.set_user_attr(
            "folds_evaluated",
            len(fold_metric_rows),
        )
        trial.set_user_attr(
            "mean_roc_auc",
            auc_summary["mean"],
        )
        trial.set_user_attr(
            "std_roc_auc",
            auc_summary["std"],
        )
        trial.set_user_attr(
            "min_roc_auc",
            auc_summary["minimum"],
        )
        trial.set_user_attr(
            "max_roc_auc",
            auc_summary["maximum"],
        )
        trial.set_user_attr(
            "mean_average_precision",
            mean_average_precision,
        )
        trial.set_user_attr(
            "policy_complete",
            bool(
                policy_summary["policy_complete"]
            ),
        )
        trial.set_user_attr(
            "policy_false_positive",
            int(
                policy_summary["false_positive"]
            ),
        )
        trial.set_user_attr(
            "policy_true_positive",
            int(
                policy_summary["true_positive"]
            ),
        )
        trial.set_user_attr(
            "policy_signals",
            int(policy_summary["signals"]),
        )
        trial.set_user_attr(
            "policy_precision",
            float(policy_summary["precision"]),
        )
        trial.set_user_attr(
            "policy_specificity",
            float(
                policy_summary["specificity"]
            ),
        )
        trial.set_user_attr(
            "policy_min_fold_specificity",
            float(
                policy_summary[
                    "min_fold_specificity"
                ]
            ),
        )
        trial.set_user_attr(
            "policy_specificity_std",
            float(
                policy_summary[
                    "std_fold_specificity"
                ]
            ),
        )
        trial.set_user_attr(
            "policy_fold_metrics",
            policy_fold_metrics.to_dict(
                orient="records"
            ),
        )
        trial.set_user_attr(
            "average_uniqueness_weighting",
            True,
        )
        trial.set_user_attr(
            "validation_metrics_weighted",
            False,
        )
        trial.set_user_attr(
            "threshold_selected",
            False,
        )

        return auc_summary["mean"]

    return objective


## 12. Run or resume 30 COMPLETE trials per tuned model

In [13]:
def trial_state_counts(
    study: optuna.Study,
) -> dict[str, int]:
    return {
        state.name: sum(
            trial.state == state
            for trial in study.trials
        )
        for state in [
            optuna.trial.TrialState.COMPLETE,
            optuna.trial.TrialState.PRUNED,
            optuna.trial.TrialState.FAIL,
            optuna.trial.TrialState.RUNNING,
            optuna.trial.TrialState.WAITING,
        ]
    }


studies = {}

for model_name in [
    "random_forest",
    "xgboost",
]:
    study = optuna.create_study(
        study_name=study_names[model_name],
        storage=storage_url,
        direction="maximize",
        sampler=build_study_sampler(),
        pruner=build_study_pruner(),
        load_if_exists=True,
    )

    counts_before = trial_state_counts(
        study
    )
    complete_before = counts_before[
        "COMPLETE"
    ]
    remaining_complete_trials = max(
        0,
        N_COMPLETE_TRIALS - complete_before,
    )

    print(
        f"\n{model_name}: "
        f"complete_before={complete_before}, "
        f"remaining_complete_trials="
        f"{remaining_complete_trials}"
    )

    if remaining_complete_trials > 0:
        study.optimize(
            create_objective(model_name),
            n_trials=remaining_complete_trials,
            n_jobs=int(
                optuna_config[
                    "optimization"
                ]["n_jobs"]
            ),
            gc_after_trial=bool(
                optuna_config[
                    "optimization"
                ]["gc_after_trial"]
            ),
            show_progress_bar=True,
        )

    counts_after = trial_state_counts(
        study
    )

    if counts_after["COMPLETE"] < (
        N_COMPLETE_TRIALS
    ):
        raise RuntimeError(
            f"{model_name} has fewer than "
            f"{N_COMPLETE_TRIALS} COMPLETE trials."
        )

    if counts_after["PRUNED"] != 0:
        raise AssertionError(
            f"{model_name} contains pruned v4 trials "
            "despite NopPruner."
        )

    if counts_after["FAIL"] != 0:
        raise AssertionError(
            f"{model_name} contains failed v4 trials. "
            "Review the study before model selection."
        )

    studies[model_name] = study

    print(
        f"{model_name} trial states:",
        counts_after,
    )
    print(
        f"{model_name} best sampler objective "
        "(mean ROC AUC):",
        study.best_value,
    )


assert (
    studies["random_forest"].sampler
    is not studies["xgboost"].sampler
)
assert isinstance(
    studies["random_forest"].pruner,
    optuna.pruners.NopPruner,
)
assert isinstance(
    studies["xgboost"].pruner,
    optuna.pruners.NopPruner,
)

print("Distinct study sampler instances: True")
print("NopPruner used for both studies: True")


e:\Iran_stock_trade\financial-ml-decision-support\.venv\Lib\site-packages\optuna\_experimental.py:31: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-07-19 17:56:24,079] A new study created in RDB with name: stage06_v4_policy_random_forest_3ac1ee3f73df



random_forest: complete_before=0, remaining_complete_trials=30


Best trial: 0. Best value: 0.516184:   3%|▎         | 1/30 [00:23<11:17, 23.37s/it]

[I 2026-07-19 17:56:47,314] Trial 0 finished with value: 0.5161844928645352 and parameters: {'n_estimators': 50, 'max_depth': 6, 'min_samples_split': 14, 'min_samples_leaf': 15, 'max_features': 'log2', 'criterion': 'gini', 'bootstrap': True, 'max_samples': 0.6910294397649198, 'class_weight_mode': 'balanced_subsample'}. Best is trial 0 with value: 0.5161844928645352.


Best trial: 0. Best value: 0.516184:   3%|▎         | 1/30 [02:22<11:17, 23.37s/it]

...

[I 2026-07-19 18:41:55,869] Trial 28 finished with value: 0.5404183066786419 and parameters: {'n_estimators': 150, 'max_depth': 2, 'learning_rate': 0.1704576403211369, 'min_child_weight': 9.464134751449988, 'subsample': 0.7710178867556868, 'colsample_bytree': 0.7526018124568483, 'gamma': 4.17737039896674, 'reg_alpha': 0.23455330877218794, 'reg_lambda': 6.34541870528128, 'max_delta_step': 4, 'class_weight_mode': 'fold_weighted_ratio'}. Best is trial 26 with value: 0.5456230605140748.


Best trial: 26. Best value: 0.545623:  97%|█████████▋| 29/30 [06:46<00:13, 13.02s/it]

[I 2026-07-19 18:42:05,773] Trial 29 finished with value: 0.5186381740637895 and parameters: {'n_estimators': 25, 'max_depth': 6, 'learning_rate': 0.10623346298601705, 'min_child_weight': 11.759754879278674, 'subsample': 0.6278960054193973, 'colsample_bytree': 0.610428371498802, 'gamma': 4.6458239099300185, 'reg_alpha': 2.270467571874403, 'reg_lambda': 0.1964914481424482, 'max_delta_step': 2, 'class_weight_mode': 'none'}. Best is trial 26 with value: 0.5456230605140748.


Best trial: 26. Best value: 0.545623: 100%|██████████| 30/30 [06:46<00:00, 13.55s/it]

xgboost trial states: {'COMPLETE': 30, 'PRUNED': 0, 'FAIL': 0, 'RUNNING': 0, 'WAITING': 0}
xgboost best sampler objective (mean ROC AUC): 0.5456230605140748
Distinct study sampler instances: True
NopPruner used for both studies: True


## 13. Select one COMPLETE trial per model family by fixed-policy hierarchy


In [14]:
selected_trials = {}
trial_ranking_frames = []
trial_export_paths = {}

for model_name, study in studies.items():
    selected_trial, ranking = (
        select_optuna_trial_by_policy(
            study
        )
    )

    selected_trials[model_name] = (
        selected_trial
    )

    ranking.insert(
        0,
        "model_name",
        model_name,
    )
    trial_ranking_frames.append(
        ranking
    )

    trials_df = study.trials_dataframe(
        attrs=(
            "number",
            "value",
            "datetime_start",
            "datetime_complete",
            "duration",
            "params",
            "user_attrs",
            "state",
        )
    )

    trials_df = trials_df.merge(
        ranking[
            [
                "trial_number",
                "policy_selection_rank",
                "selected_by_policy_hierarchy",
            ]
        ],
        left_on="number",
        right_on="trial_number",
        how="left",
        validate="one_to_one",
    ).drop(columns=["trial_number"])

    trial_path = (
        RESULT_PATHS["optuna"]
        / f"06_{model_name}_trials.csv"
    )
    ranking_path = (
        RESULT_PATHS["optuna"]
        / (
            f"06_{model_name}_"
            "trial_policy_selection_ranking.csv"
        )
    )

    trials_df.to_csv(
        trial_path,
        index=False,
        encoding="utf-8-sig",
    )
    ranking.to_csv(
        ranking_path,
        index=False,
        encoding="utf-8-sig",
    )

    trial_export_paths[model_name] = {
        "trials": trial_path,
        "ranking": ranking_path,
    }

    print(
        f"\n{model_name} selected trial:",
        selected_trial.number,
    )
    print(
        "  false positives:",
        selected_trial.user_attrs[
            "policy_false_positive"
        ],
    )
    print(
        "  precision:",
        selected_trial.user_attrs[
            "policy_precision"
        ],
    )
    print(
        "  minimum fold specificity:",
        selected_trial.user_attrs[
            "policy_min_fold_specificity"
        ],
    )
    print(
        "  mean ROC AUC:",
        selected_trial.user_attrs[
            "mean_roc_auc"
        ],
    )
    print(
        "  params:",
        selected_trial.params,
    )

all_trial_rankings_df = pd.concat(
    trial_ranking_frames,
    ignore_index=True,
)

display(
    all_trial_rankings_df.loc[
        all_trial_rankings_df[
            "policy_selection_rank"
        ].notna()
    ].sort_values(
        ["model_name", "policy_selection_rank"],
        kind="stable",
    )
)



random_forest selected trial: 8
  false positives: 1079
  precision: 0.6422413793103449
  minimum fold specificity: 0.9333333333333333
  mean ROC AUC: 0.5039272564036843
  params: {'n_estimators': 50, 'max_depth': 9, 'min_samples_split': 22, 'min_samples_leaf': 11, 'max_features': 0.5, 'criterion': 'log_loss', 'bootstrap': True, 'max_samples': 0.9971658557508559, 'class_weight_mode': 'none'}

xgboost selected trial: 10
  false positives: 1029
  precision: 0.6588196286472149
  minimum fold specificity: 0.9445887445887445
  mean ROC AUC: 0.5227934492043504
  params: {'n_estimators': 100, 'max_depth': 8, 'learning_rate': 0.018637302558129572, 'min_child_weight': 3.1732600961867776, 'subsample': 0.6914421358691342, 'colsample_bytree': 0.5877906125992687, 'gamma': 2.6758232429856275, 'reg_alpha': 0.06038497631137781, 'reg_lambda': 48.34276122213323, 'max_delta_step': 0, 'class_weight_mode': 'fold_weighted_ratio'}


,model_name,trial_number,objective_value,policy_complete,false_positive,precision,min_fold_specificity,std_fold_specificity,mean_average_precision,mean_roc_auc,policy_selection_rank,selected_by_policy_hierarchy
0,random_forest,8,0.503927,True,1079,0.642241,0.933333,0.007232,0.611765,0.503927,1,True
1,random_forest,9,0.509678,True,1080,0.641910,0.935931,0.005825,0.612057,0.509678,2,False
2,random_forest,1,0.519243,True,1085,0.640252,0.939394,0.004289,0.619447,0.519243,3,False
3,random_forest,6,0.513831,True,1088,0.639257,0.935931,0.006233,0.617820,0.513831,4,False
4,random_forest,4,0.516511,True,1098,0.635942,0.939394,0.003920,0.613052,0.516511,5,False
5,random_forest,24,0.520763,True,1102,0.634615,0.937662,0.005278,0.625844,0.520763,6,False
6,random_forest,23,0.520947,True,1103,0.634284,0.936797,0.005057,0.625979,0.520947,7,False
7,random_forest,3,0.516509,True,1103,0.634284,0.935931,0.005728,0.616629,0.516509,8,False
8,random_forest,18,0.509588,True,1103,0.634284,0.930736,0.007395,0.617088,0.509588,9,False
9,random_forest,29,0.519164,True,1112,0.631300,0.933333,0.007248,0.625541,0.519164,10,False


## 14. Evaluate selected model-family parameters across all feature sets

The selected family hyperparameters are frozen. Each tree family is now
re-estimated on every fold for each of the nine pre-registered feature sets.
This is an ablation and operational policy comparison, not nine additional
Optuna searches.


In [15]:
best_params = {
    model_name: dict(
        trial.params
    )
    for model_name, trial in (
        selected_trials.items()
    )
}

tuned_fold_metric_rows = []
tuned_prediction_parts = []

for model_name in [
    "random_forest",
    "xgboost",
]:
    print(
        f"\nSelected model family: "
        f"{model_name}"
    )

    for feature_set_name in FEATURE_SETS:
        feature_set_metric_rows = []

        for fold in folds:
            metrics, predictions = fit_predict_fold(
                model_name=model_name,
                feature_set_name=feature_set_name,
                fold=fold,
                params=best_params[
                    model_name
                ],
                return_predictions=True,
            )

            tuned_fold_metric_rows.append(
                metrics
            )
            tuned_prediction_parts.append(
                predictions
            )
            feature_set_metric_rows.append(
                metrics
            )

        print(
            f"  {feature_set_name}: "
            f"features={len(FEATURE_SETS[feature_set_name])} "
            f"mean_auc="
            f"{np.mean([row['roc_auc'] for row in feature_set_metric_rows]):.6f} "
            f"mean_AP="
            f"{np.mean([row['average_precision'] for row in feature_set_metric_rows]):.6f}"
        )

tuned_fold_metrics_df = pd.DataFrame(
    tuned_fold_metric_rows
)
tuned_predictions_df = pd.concat(
    tuned_prediction_parts,
    ignore_index=True,
)

all_fold_metrics_df = pd.concat(
    [
        baseline_fold_metrics_df,
        tuned_fold_metrics_df,
    ],
    ignore_index=True,
)

all_predictions_df = pd.concat(
    [
        baseline_predictions_df,
        tuned_predictions_df,
    ],
    ignore_index=True,
)

all_fold_metrics_df.to_csv(
    RESULT_PATHS["metrics"]
    / "06_model_selection_fold_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

all_predictions_df.to_csv(
    RESULT_PATHS["predictions"]
    / "06_walk_forward_validation_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

display(
    tuned_fold_metrics_df.groupby(
        ["model_name", "feature_set_name"],
        sort=False,
    )[["roc_auc", "average_precision"]]
    .mean()
    .reset_index()
)



Selected model family: random_forest
  A_baseline_35: features=35 mean_auc=0.510761 mean_AP=0.617855
  B_plus_started: features=36 mean_auc=0.502354 mean_AP=0.613037
  C_plus_breadth_raw: features=36 mean_auc=0.501434 mean_AP=0.612850
  D_plus_breadth_ema30: features=36 mean_auc=0.503814 mean_AP=0.610352
  E_plus_breadth_slope5: features=36 mean_auc=0.504352 mean_AP=0.612706
  F_plus_all_continuous_breadth: features=38 mean_auc=0.519093 mean_AP=0.620274
  G_plus_breadth_regime: features=36 mean_auc=0.511531 mean_AP=0.616568
  H_plus_slope5_and_regime: features=37 mean_auc=0.502679 mean_AP=0.611205
  I_full_40: features=40 mean_auc=0.503927 mean_AP=0.611765

Selected model family: xgboost
  A_baseline_35: features=35 mean_auc=0.510648 mean_AP=0.617057
  B_plus_started: features=36 mean_auc=0.503358 mean_AP=0.612437
  C_plus_breadth_raw: features=36 mean_auc=0.508884 mean_AP=0.614734
  D_plus_breadth_ema30: features=36 mean_auc=0.506245 mean_AP=0.614995
  E_plus_breadth_slope5: features

,model_name,feature_set_name,roc_auc,average_precision
0,random_forest,A_baseline_35,0.510761,0.617855
1,random_forest,B_plus_started,0.502354,0.613037
2,random_forest,C_plus_breadth_raw,0.501434,0.612850
3,random_forest,D_plus_breadth_ema30,0.503814,0.610352
4,random_forest,E_plus_breadth_slope5,0.504352,0.612706
5,random_forest,F_plus_all_continuous_breadth,0.519093,0.620274
6,random_forest,G_plus_breadth_regime,0.511531,0.616568
7,random_forest,H_plus_slope5_and_regime,0.502679,0.611205
8,random_forest,I_full_40,0.503927,0.611765
9,xgboost,A_baseline_35,0.510648,0.617057


## 15. Fixed-policy comparison and final model/feature-set decision

For every tree model and feature set, apply the same daily Top 5% quota.
Selection is hierarchical and begins with total false positives. ROC AUC is
reported only after policy error type, precision, and fold stability.


In [16]:
def aggregate_variant_fold_metrics(
    fold_metrics: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for (
        model_name,
        feature_set_name,
    ), group in fold_metrics.groupby(
        ["model_name", "feature_set_name"],
        sort=False,
        observed=False,
    ):
        rows.append(
            {
                "model_name": model_name,
                "feature_set_name": feature_set_name,
                "folds": int(
                    group["fold_id"].nunique()
                ),
                "feature_count": int(
                    group["feature_count"].iloc[0]
                ),
                "mean_roc_auc": float(
                    group["roc_auc"].mean()
                ),
                "std_roc_auc": float(
                    group["roc_auc"].std(ddof=0)
                ),
                "min_roc_auc": float(
                    group["roc_auc"].min()
                ),
                "mean_average_precision": float(
                    group[
                        "average_precision"
                    ].mean()
                ),
                "mean_log_loss": float(
                    group["log_loss"].mean()
                ),
                "mean_brier_score": float(
                    group["brier_score"].mean()
                ),
                "mean_specificity_at_0_50": float(
                    group["specificity"].mean()
                ),
                "mean_sensitivity_at_0_50": float(
                    group["sensitivity"].mean()
                ),
                "total_fit_seconds": float(
                    group["fit_seconds"].sum()
                ),
                "total_inference_seconds": float(
                    group[
                        "inference_seconds"
                    ].sum()
                ),
            }
        )

    return pd.DataFrame(rows)


model_selection_summary_df = (
    aggregate_variant_fold_metrics(
        all_fold_metrics_df
    )
)

tree_variant_summary_df = (
    aggregate_variant_fold_metrics(
        tuned_fold_metrics_df
    )
)

policy_summary_rows = []
policy_fold_parts = []
policy_assignment_parts = []

for (
    model_name,
    feature_set_name,
), group in tuned_predictions_df.groupby(
    ["model_name", "feature_set_name"],
    sort=False,
    observed=False,
):
    assignments = (
        apply_daily_top_fraction_policy(
            group,
            policy=PRIMARY_POLICY,
        )
    )
    summary, fold_policy = (
        summarize_policy_predictions(
            assignments,
            policy=PRIMARY_POLICY,
        )
    )

    predictive_row = (
        tree_variant_summary_df.loc[
            tree_variant_summary_df[
                "model_name"
            ].eq(model_name)
            & tree_variant_summary_df[
                "feature_set_name"
            ].eq(feature_set_name)
        ].iloc[0]
    )

    policy_summary_rows.append(
        {
            "model_name": model_name,
            "feature_set_name": feature_set_name,
            "feature_count": int(
                len(FEATURE_SETS[feature_set_name])
            ),
            **{
                key: value
                for key, value in summary.items()
                if key != "policy"
            },
            "mean_roc_auc": float(
                predictive_row["mean_roc_auc"]
            ),
            "std_roc_auc": float(
                predictive_row["std_roc_auc"]
            ),
            "min_roc_auc": float(
                predictive_row["min_roc_auc"]
            ),
            "mean_average_precision": float(
                predictive_row[
                    "mean_average_precision"
                ]
            ),
        }
    )

    fold_policy.insert(
        0,
        "feature_set_name",
        feature_set_name,
    )
    fold_policy.insert(
        0,
        "model_name",
        model_name,
    )
    policy_fold_parts.append(fold_policy)
    policy_assignment_parts.append(assignments)

policy_summary_df = pd.DataFrame(
    policy_summary_rows
)
policy_fold_metrics_df = pd.concat(
    policy_fold_parts,
    ignore_index=True,
)
policy_assignments_df = pd.concat(
    policy_assignment_parts,
    ignore_index=True,
)

policy_ranking_df = rank_policy_candidates(
    policy_summary_df
)

selected_policy_row = (
    policy_ranking_df.loc[
        policy_ranking_df[
            "selected_by_policy_hierarchy"
        ]
    ].iloc[0]
)

PRIMARY_SELECTED_MODEL = str(
    selected_policy_row["model_name"]
)
PRIMARY_SELECTED_FEATURE_SET = str(
    selected_policy_row[
        "feature_set_name"
    ]
)
PRIMARY_SELECTED_FEATURES = FEATURE_SETS[
    PRIMARY_SELECTED_FEATURE_SET
]

challenger_policy_row = (
    policy_ranking_df.loc[
        policy_ranking_df[
            "policy_selection_rank"
        ].eq(2)
    ].iloc[0]
)
CHALLENGER_MODEL = str(
    challenger_policy_row["model_name"]
)
CHALLENGER_FEATURE_SET = str(
    challenger_policy_row[
        "feature_set_name"
    ]
)

logistic_row = (
    model_selection_summary_df.loc[
        model_selection_summary_df[
            "model_name"
        ].eq("logistic_regression")
    ].iloc[0]
)

selection_decision = {
    "primary_selected_model": (
        PRIMARY_SELECTED_MODEL
    ),
    "primary_selected_feature_set": (
        PRIMARY_SELECTED_FEATURE_SET
    ),
    "primary_selected_features": (
        PRIMARY_SELECTED_FEATURES
    ),
    "challenger_model": CHALLENGER_MODEL,
    "challenger_feature_set": (
        CHALLENGER_FEATURE_SET
    ),
    "selection_scope": {
        "models": [
            "random_forest",
            "xgboost",
        ],
        "feature_sets": list(
            FEATURE_SETS
        ),
    },
    "tuning_feature_set": (
        TUNING_FEATURE_SET_NAME
    ),
    "tune_each_feature_set_separately": (
        False
    ),
    "optuna_sampler_objective": (
        "equal_fold_mean_roc_auc"
    ),
    "trial_selection_hierarchy": (
        stage06_config[
            "trial_selection_hierarchy"
        ]
    ),
    "model_feature_selection_hierarchy": (
        stage06_config[
            "model_feature_selection_hierarchy"
        ]
    ),
    "fixed_policy": {
        "type": (
            "daily_cross_sectional_top_fraction"
        ),
        "selected_fraction": float(
            PRIMARY_POLICY.fraction
        ),
        "minimum_signals_per_date": int(
            PRIMARY_POLICY.minimum_signals_per_date
        ),
        "quota_rounding": "ceil",
        "tie_break": [
            "probability_positive_descending",
            "symbol_ascending",
            "event_id_ascending",
        ],
    },
    "threshold_selected": False,
    "selected_false_positive": int(
        selected_policy_row[
            "false_positive"
        ]
    ),
    "selected_true_positive": int(
        selected_policy_row[
            "true_positive"
        ]
    ),
    "selected_precision": float(
        selected_policy_row["precision"]
    ),
    "selected_specificity": float(
        selected_policy_row["specificity"]
    ),
    "selected_min_fold_specificity": float(
        selected_policy_row[
            "min_fold_specificity"
        ]
    ),
    "selected_mean_roc_auc": float(
        selected_policy_row["mean_roc_auc"]
    ),
    "selected_mean_average_precision": float(
        selected_policy_row[
            "mean_average_precision"
        ]
    ),
    "selected_tree_beats_logistic_on_mean_auc": bool(
        float(
            selected_policy_row[
                "mean_roc_auc"
            ]
        )
        > float(
            logistic_row["mean_roc_auc"]
        )
    ),
    "selected_trial_numbers": {
        model_name: int(
            trial.number
        )
        for model_name, trial in (
            selected_trials.items()
        )
    },
    "average_uniqueness_weighting": {
        "method": "average_uniqueness",
        "scope": (
            "current_fold_training_events_only"
        ),
        "concurrency": "within_symbol",
        "weighted_models": sorted(
            WEIGHTED_MODELS
        ),
        "dummy_prior_weighted": False,
        "validation_metrics_weighted": False,
    },
    "hard_started_filter_applied": False,
    "hard_breadth_filter_applied": False,
    "unseen_test_used": False,
}

model_selection_summary_df.to_csv(
    RESULT_PATHS["metrics"]
    / "06_model_selection_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

policy_summary_df.to_csv(
    RESULT_PATHS["metrics"]
    / "06_feature_set_policy_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
policy_fold_metrics_df.to_csv(
    RESULT_PATHS["metrics"]
    / "06_feature_set_policy_fold_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)
policy_ranking_df.to_csv(
    RESULT_PATHS["metrics"]
    / "06_feature_set_policy_ranking.csv",
    index=False,
    encoding="utf-8-sig",
)
policy_assignments_df.to_csv(
    RESULT_PATHS["predictions"]
    / "06_feature_set_daily_policy_assignments.csv",
    index=False,
    encoding="utf-8-sig",
)

(
    RESULT_PATHS["manifests"]
    / "06_model_selection_decision.json"
).write_text(
    json.dumps(
        selection_decision,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

best_hyperparameters_payload = {
    "study_signature": STUDY_SIGNATURE,
    "random_seed": SEED,
    "tuning_feature_set": (
        TUNING_FEATURE_SET_NAME
    ),
    "selected_model": (
        PRIMARY_SELECTED_MODEL
    ),
    "selected_feature_set": (
        PRIMARY_SELECTED_FEATURE_SET
    ),
    "selected_features": (
        PRIMARY_SELECTED_FEATURES
    ),
    "trial_selection_rule": (
        stage06_config[
            "trial_selection_hierarchy"
        ]
    ),
}

for model_name, trial in (
    selected_trials.items()
):
    best_hyperparameters_payload[
        model_name
    ] = {
        "selected_trial_number": int(
            trial.number
        ),
        "policy_false_positive": int(
            trial.user_attrs[
                "policy_false_positive"
            ]
        ),
        "policy_precision": float(
            trial.user_attrs[
                "policy_precision"
            ]
        ),
        "mean_roc_auc": float(
            trial.user_attrs[
                "mean_roc_auc"
            ]
        ),
        "params": dict(
            trial.params
        ),
    }

(
    RESULT_PATHS["optuna"]
    / "06_best_hyperparameters.json"
).write_text(
    json.dumps(
        best_hyperparameters_payload,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

display(
    policy_ranking_df[
        [
            "policy_selection_rank",
            "model_name",
            "feature_set_name",
            "feature_count",
            "signals",
            "true_positive",
            "false_positive",
            "precision",
            "specificity",
            "min_fold_specificity",
            "mean_average_precision",
            "mean_roc_auc",
        ]
    ].head(18)
)

print(
    "Primary selected model:",
    PRIMARY_SELECTED_MODEL,
)
print(
    "Primary selected feature set:",
    PRIMARY_SELECTED_FEATURE_SET,
)
print(
    "Selected false positives:",
    selection_decision[
        "selected_false_positive"
    ],
)
print(
    "Selected precision:",
    selection_decision[
        "selected_precision"
    ],
)
print(
    "Challenger:",
    CHALLENGER_MODEL,
    CHALLENGER_FEATURE_SET,
)


,policy_selection_rank,model_name,feature_set_name,feature_count,signals,true_positive,false_positive,precision,specificity,min_fold_specificity,mean_average_precision,mean_roc_auc
0,1,xgboost,I_full_40,40,3016,1987,1029,0.658820,0.951040,0.944589,0.622358,0.522793
1,2,xgboost,E_plus_breadth_slope5,36,3016,1960,1056,0.649867,0.949755,0.946320,0.617042,0.512905
2,3,xgboost,F_plus_all_continuous_breadth,38,3016,1959,1057,0.649536,0.949707,0.946320,0.617923,0.512568
3,4,xgboost,H_plus_slope5_and_regime,37,3016,1955,1061,0.648210,0.949517,0.947592,0.617385,0.517169
4,5,random_forest,D_plus_breadth_ema30,36,3016,1955,1061,0.648210,0.949517,0.935931,0.610352,0.503814
5,6,xgboost,G_plus_breadth_regime,36,3016,1954,1062,0.647878,0.949469,0.944589,0.611834,0.513604
6,7,xgboost,A_baseline_35,35,3016,1950,1066,0.646552,0.949279,0.941126,0.617057,0.510648
7,8,xgboost,C_plus_breadth_raw,36,3016,1948,1068,0.645889,0.949184,0.941126,0.614734,0.508884
8,9,random_forest,F_plus_all_continuous_breadth,38,3016,1947,1069,0.645557,0.949136,0.935931,0.620274,0.519093
9,10,xgboost,B_plus_started,36,3016,1946,1070,0.645225,0.949089,0.942857,0.612437,0.503358


Primary selected model: xgboost
Primary selected feature set: I_full_40
Selected false positives: 1029
Selected precision: 0.6588196286472149
Challenger: xgboost E_plus_breadth_slope5


## 16. Freeze Stage 06 v4 policy-first manifest


In [17]:
manifest = {
    "stage": 6,
    "status": (
        "frozen_after_integrity_checks"
    ),
    "stage_version": (
        "v4_breadth_policy_first_model_feature_selection"
    ),
    "notebook": (
        "06_optuna_model_selection.ipynb"
    ),
    "git_commit_sha": git_commit_sha(
        REPOSITORY_ROOT
    ),
    "software": software_manifest(),
    "study_signature": STUDY_SIGNATURE,
    "random_seed": SEED,
    "input_population": {
        "candidate_events": int(
            len(candidate_panel)
        ),
        "symbols": int(
            candidate_panel[
                "symbol"
            ].nunique()
        ),
        "features": MODEL_FEATURES,
        "numeric_features": (
            NUMERIC_FEATURES
        ),
        "categorical_features": (
            CATEGORICAL_FEATURES
        ),
        "baseline_feature_count": int(
            len(BASELINE_FEATURES)
        ),
        "added_stage04_features": (
            ADDED_STAGE04_FEATURES
        ),
        "stage04_primary_threshold_fraction": (
            0.15
        ),
    },
    "feature_set_design": {
        "feature_sets": FEATURE_SETS,
        "tuning_feature_set": (
            TUNING_FEATURE_SET_NAME
        ),
        "tune_each_feature_set_separately": (
            False
        ),
        "shared_family_hyperparameters": (
            True
        ),
    },
    "validation": {
        "source": (
            "frozen Stage 05 v2 folds"
        ),
        "stage05_code_commit_sha": (
            stage05_manifest[
                "git_commit_sha"
            ]
        ),
        "stage05_configuration_hash": (
            stage05_manifest[
                "configuration_hash"
            ]
        ),
        "fold_design_schema_version": (
            stage05_manifest[
                "fold_design_schema_version"
            ]
        ),
        "folds": int(len(folds)),
        "method": (
            "purged_anchored_walk_forward"
        ),
        "fold_weighting": "equal",
        "optuna_sampler_objective": (
            "mean_fold_roc_auc"
        ),
        "diagnostic_threshold": (
            DIAGNOSTIC_THRESHOLD
        ),
        "probability_threshold_selected": (
            False
        ),
    },
    "fixed_signal_policy": {
        "schema_version": (
            POLICY_SELECTION_SCHEMA_VERSION
        ),
        "policy": {
            "type": (
                "daily_cross_sectional_top_fraction"
            ),
            "selected_fraction": float(
                PRIMARY_POLICY.fraction
            ),
            "minimum_signals_per_date": int(
                PRIMARY_POLICY.minimum_signals_per_date
            ),
            "quota_rounding": "ceil",
            "score_order": "descending",
            "tie_break": [
                "symbol_ascending",
                "event_id_ascending",
            ],
        },
        "trial_selection_hierarchy": (
            stage06_config[
                "trial_selection_hierarchy"
            ]
        ),
        "model_feature_selection_hierarchy": (
            stage06_config[
                "model_feature_selection_hierarchy"
            ]
        ),
    },
    "sample_weighting": {
        "schema_version": (
            AVERAGE_UNIQUENESS_SCHEMA_VERSION
        ),
        "method": "average_uniqueness",
        "source_scope": (
            "current_fold_training_events_only"
        ),
        "concurrency_scope": (
            "within_symbol"
        ),
        "active_interval": (
            "inclusive_start_and_end"
        ),
        "normalization": (
            "fold_train_mean_one"
        ),
        "weighted_models": sorted(
            WEIGHTED_MODELS
        ),
        "dummy_prior_weighted": False,
        "validation_events_used": False,
        "validation_metrics_weighted": (
            False
        ),
        "return_attribution_multiplier_used": (
            False
        ),
        "sequential_bootstrap_used": False,
        "fold_audit": (
            average_uniqueness_audit_df.to_dict(
                orient="records"
            )
        ),
    },
    "optuna": {
        "required_complete_trials_per_model": (
            N_COMPLETE_TRIALS
        ),
        "pruner": "NopPruner",
        "every_trial_evaluates_all_folds": (
            True
        ),
        "study_names": study_names,
        "database_file": str(
            database_path
        ),
        "trial_state_counts": {
            model_name: trial_state_counts(
                study
            )
            for model_name, study in (
                studies.items()
            )
        },
    },
    "best_hyperparameters": (
        best_hyperparameters_payload
    ),
    "selection_decision": (
        selection_decision
    ),
    "unseen_test_used": False,
    "configuration_hash": stable_object_hash(
        {
            "optuna": optuna_config,
            "models": models_config,
            "stage06": stage06_config,
            "feature_manifest": (
                feature_manifest_df.to_dict(
                    orient="records"
                )
            ),
            "fold_boundaries": (
                fold_boundaries_df.to_dict(
                    orient="records"
                )
            ),
            "average_uniqueness_audit": (
                average_uniqueness_audit_df.to_dict(
                    orient="records"
                )
            ),
        }
    ),
}

manifest_path = (
    RESULT_PATHS["manifests"]
    / "06_optuna_model_selection_manifest.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Manifest:", manifest_path)
print(
    "Manifest code SHA:",
    manifest["git_commit_sha"],
)


Manifest: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\06_optuna_model_selection_manifest.json
Manifest code SHA: 9928795ae10f0a9fd64788e1880b6cacc249e855


## 17. Final integrity checks

In [18]:
assert candidate_errors_df.empty
assert calendar_errors_df.empty
assert fold_reproduction_audit_df[
    "exact_membership_match"
].all()

assert len(candidate_panel) == 118_464
assert len(MODEL_FEATURES) == 40
assert len(NUMERIC_FEATURES) == 38
assert CATEGORICAL_FEATURES == [
    "gmma_state",
    "market_breadth_regime",
]
assert len(BASELINE_FEATURES) == 35
assert len(FEATURE_SETS) == 9
assert len(folds) == 5

assert average_uniqueness_audit_df[
    "validation_events_used"
].eq(0).all()
assert average_uniqueness_audit_df[
    "nonfinite_raw_uniqueness"
].eq(0).all()
assert average_uniqueness_audit_df[
    "nonfinite_sample_weight"
].eq(0).all()
assert average_uniqueness_audit_df[
    "nonpositive_sample_weight"
].eq(0).all()
assert np.allclose(
    average_uniqueness_audit_df[
        "mean_sample_weight"
    ].to_numpy(dtype=float),
    1.0,
    atol=1.0e-12,
    rtol=1.0e-12,
)

expected_weight_rows = int(
    fold_reproduction_audit_df[
        "reproduced_train_events"
    ].sum()
)
assert len(
    all_fold_train_weights_df
) == expected_weight_rows

assert not all_fold_train_weights_df[
    [
        "fold_id",
        "event_id",
    ]
].duplicated().any()

assert (
    baseline_fold_metrics_df.loc[
        baseline_fold_metrics_df[
            "model_name"
        ].eq("dummy_prior"),
        "average_uniqueness_weighting_applied",
    ].eq(False).all()
)
assert (
    baseline_fold_metrics_df.loc[
        baseline_fold_metrics_df[
            "model_name"
        ].eq("logistic_regression"),
        "average_uniqueness_weighting_applied",
    ].eq(True).all()
)
assert tuned_fold_metrics_df[
    "average_uniqueness_weighting_applied"
].eq(True).all()
assert all_fold_metrics_df[
    "validation_metrics_weighted"
].eq(False).all()

assert tuned_fold_metrics_df.groupby(
    ["model_name", "feature_set_name"]
)["fold_id"].nunique().eq(5).all()

assert set(
    tuned_fold_metrics_df[
        "model_name"
    ]
) == {
    "random_forest",
    "xgboost",
}
assert set(
    tuned_fold_metrics_df[
        "feature_set_name"
    ]
) == set(FEATURE_SETS)

assert all_fold_metrics_df[
    [
        "roc_auc",
        "average_precision",
        "log_loss",
        "brier_score",
        "balanced_accuracy",
        "specificity",
        "sensitivity",
        "precision",
        "f1",
        "mcc",
    ]
].notna().all().all()

assert all_predictions_df[
    "probability_positive"
].between(0.0, 1.0).all()
assert all_predictions_df[
    "validation_metric_weight"
].eq(1.0).all()

assert not all_predictions_df[
    [
        "model_name",
        "feature_set_name",
        "fold_id",
        "event_id",
    ]
].duplicated().any()

for model_name, study in (
    studies.items()
):
    counts = trial_state_counts(study)

    assert counts["COMPLETE"] >= (
        N_COMPLETE_TRIALS
    )
    assert counts["PRUNED"] == 0
    assert counts["FAIL"] == 0

    for trial in study.trials:
        if trial.state == (
            optuna.trial.TrialState.COMPLETE
        ):
            assert trial.user_attrs[
                "folds_evaluated"
            ] == 5
            assert trial.user_attrs[
                "policy_complete"
            ] is True
            assert trial.user_attrs[
                "threshold_selected"
            ] is False

assert isinstance(
    studies["random_forest"].pruner,
    optuna.pruners.NopPruner,
)
assert isinstance(
    studies["xgboost"].pruner,
    optuna.pruners.NopPruner,
)
assert (
    studies["random_forest"].sampler
    is not studies["xgboost"].sampler
)

for model_name, trial in (
    selected_trials.items()
):
    ranking = all_trial_rankings_df.loc[
        all_trial_rankings_df[
            "model_name"
        ].eq(model_name)
    ]
    selected_row = ranking.loc[
        ranking[
            "selected_by_policy_hierarchy"
        ]
    ].iloc[0]

    assert int(
        selected_row[
            "trial_number"
        ]
    ) == int(trial.number)
    assert bool(
        selected_row["policy_complete"]
    )

assert len(policy_summary_df) == (
    2 * len(FEATURE_SETS)
)
assert policy_summary_df[
    "policy_complete"
].all()
assert policy_summary_df[
    "signals"
].nunique() == 1
assert policy_summary_df[
    "dates_with_signal"
].eq(
    policy_summary_df["dates"]
).all()

assert int(
    policy_ranking_df[
        "selected_by_policy_hierarchy"
    ].sum()
) == 1
assert (
    PRIMARY_SELECTED_FEATURE_SET
    in FEATURE_SETS
)
assert (
    PRIMARY_SELECTED_MODEL
    in {"random_forest", "xgboost"}
)
assert selection_decision[
    "threshold_selected"
] is False
assert selection_decision[
    "hard_started_filter_applied"
] is False
assert selection_decision[
    "hard_breadth_filter_applied"
] is False
assert selection_decision[
    "unseen_test_used"
] is False
assert manifest[
    "unseen_test_used"
] is False

print("Notebook 06 integrity checks: PASSED")
print(
    "Candidate events:",
    len(candidate_panel),
)
print(
    "Final pooled-model features:",
    len(MODEL_FEATURES),
)
print(
    "Pre-registered feature sets:",
    len(FEATURE_SETS),
)
print(
    "Frozen Stage 05 folds:",
    len(folds),
)
print(
    "Average Uniqueness weighting:",
    "fold-local within-symbol",
)
print(
    "COMPLETE trials per tuned model:",
    N_COMPLETE_TRIALS,
)
print(
    "Optuna sampler objective:",
    "equal-fold mean ROC AUC",
)
print(
    "Trial selection priority:",
    "minimum false positives under fixed Top 5% policy",
)
print(
    "Final selection priority:",
    "minimum false positives, then precision and fold stability",
)
print(
    "Threshold selection performed: False"
)
print(
    "Hard started filter applied: False"
)
print(
    "Hard breadth filter applied: False"
)
print(
    "Primary selected model:",
    PRIMARY_SELECTED_MODEL,
)
print(
    "Primary selected feature set:",
    PRIMARY_SELECTED_FEATURE_SET,
)
print(
    "Selected signals:",
    int(selected_policy_row["signals"]),
)
print(
    "Selected false positives:",
    int(selected_policy_row["false_positive"]),
)
print(
    "Selected precision:",
    float(selected_policy_row["precision"]),
)
print(
    "Unseen-test used in Stage 06 decisions: False"
)
print(
    "Next stage: Notebook 07 — train the frozen selected "
    "model and selected feature set on all eligible train events."
)


Notebook 06 integrity checks: PASSED
Candidate events: 118464
Final pooled-model features: 40
Pre-registered feature sets: 9
Frozen Stage 05 folds: 5
Average Uniqueness weighting: fold-local within-symbol
COMPLETE trials per tuned model: 30
Optuna sampler objective: equal-fold mean ROC AUC
Trial selection priority: minimum false positives under fixed Top 5% policy
Final selection priority: minimum false positives, then precision and fold stability
Threshold selection performed: False
Hard started filter applied: False
Hard breadth filter applied: False
Primary selected model: xgboost
Primary selected feature set: I_full_40
Selected signals: 3016
Selected false positives: 1029
Selected precision: 0.6588196286472149
Unseen-test used in Stage 06 decisions: False
Next stage: Notebook 07 — train the frozen selected model and selected feature set on all eligible train events.


## Review before freezing Stage 06

Inspect:

- `results/audits/06_average_uniqueness_audit.csv`
- `results/audits/06_average_uniqueness_symbol_audit.csv`
- `results/folds/06_fold_train_average_uniqueness_weights.csv`
- `results/optuna/06_random_forest_trials.csv`
- `results/optuna/06_xgboost_trials.csv`
- `results/optuna/06_random_forest_trial_policy_selection_ranking.csv`
- `results/optuna/06_xgboost_trial_policy_selection_ranking.csv`
- `results/optuna/06_best_hyperparameters.json`
- `results/metrics/06_model_selection_fold_metrics.csv`
- `results/metrics/06_model_selection_summary.csv`
- `results/metrics/06_feature_set_policy_summary.csv`
- `results/metrics/06_feature_set_policy_fold_metrics.csv`
- `results/metrics/06_feature_set_policy_ranking.csv`
- `results/predictions/06_walk_forward_validation_predictions.csv`
- `results/predictions/06_feature_set_daily_policy_assignments.csv`
- `results/manifests/06_model_selection_decision.json`
- `results/manifests/06_optuna_model_selection_manifest.json`
- `results/audits/06_fold_reproduction_audit.csv`
- `results/audits/06_candidate_panel_audit.csv`

Notebook 07 must consume both the frozen selected model family and the frozen
selected feature set. The unseen test remains unopened.
